# Scraping Data Ulasan Play Store (Indonesia)

Notebook ini mengambil data ulasan aplikasi Duolingo secara mandiri dari internet menggunakan `google-play-scraper`.

Target:
- Minimal 10.000 sampel
- Data disimpan ke CSV untuk dipakai pada notebook analisis sentimen

Sumber data:
- Google Play Store

In [ ]:
%pip install -q google-play-scraper pandas tqdm

In [ ]:
import warnings
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from google_play_scraper import Sort, reviews

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

# Scrapping review aplikasi Duolingo
APP_ID = "com.duolingo"
APP_NAME = "DUOLINGO"
LANG = "id"
COUNTRY = "id"
TARGET_COUNT = 20000
MAX_BATCH = 200

OUTPUT_DIR = Path("data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / "reviews_duolingo.csv"

print(f"Target total sampel: ~{TARGET_COUNT}".replace(",", "."))

Target total sampel: ~20000


In [ ]:
def scrape_reviews_for_app(app_id: str, app_name: str, target_count: int = 2000):
    """Scrape review sampai target terpenuhi atau data habis."""
    all_rows = []
    token = None

    while len(all_rows) < target_count:
        batch_count = min(MAX_BATCH, target_count - len(all_rows))
        result, token = reviews(
            app_id,
            lang=LANG,
            country=COUNTRY,
            sort=Sort.NEWEST,
            count=batch_count,
            continuation_token=token,
        )

        if not result:
            break

        for r in result:
            all_rows.append(
                {
                    "app_id": app_id,
                    "app_name": app_name,
                    "review_id": r.get("reviewId"),
                    "user_name": r.get("userName"),
                    "score": r.get("score"),
                    "content": r.get("content"),
                    "thumbs_up": r.get("thumbsUpCount"),
                    "at": r.get("at"),
                    "app_version": r.get("appVersion"),
                }
            )

        if token is None:
            break

    return all_rows

In [ ]:
all_reviews = scrape_reviews_for_app(APP_ID, APP_NAME, target_count=TARGET_COUNT)

raw_df = pd.DataFrame(all_reviews)
print(f"\nTotal ulasan terkumpul (raw): {len(raw_df):,}".replace(",", "."))
raw_df.head()


Total ulasan terkumpul (raw): 20.000


,app_id,app_name,review_id,user_name,score,content,thumbs_up,at,app_version
0,com.duolingo,DUOLINGO,7852f726-11f1-4de3-8bab-1add6f09d7f7,Farhan Abdillah,5,gak ada update aku versi old,0,2026-04-26 13:24:18,5.121.14
1,com.duolingo,DUOLINGO,4ba770cc-55dc-41cc-bd78-04524bf94c98,Devi Melani,3,"Why is this app so slow? Previously, I was studying smoothly without any problems, but now, just opening a single le...",0,2026-04-26 13:11:49,None
2,com.duolingo,DUOLINGO,3652eaa4-5f07-4c5f-b79a-b6bcb9a1bbb6,Nun_azzahh,5,"Duolingo ngebantu banget , asalnya aku main duolingo main belajar di duolingo cuman buat b inggris trs aku ikut esku...",0,2026-04-26 12:49:42,6.71.2
3,com.duolingo,DUOLINGO,435b2c23-4849-41e6-9f65-54117fc4950c,Sukarti 12,5,bagus sih ini saya jadi bisa belajar bahasa Inggris tapi ada yang aneh pas aku pake tulisan premium malah salah koca...,0,2026-04-26 11:44:12,6.76.3
4,com.duolingo,DUOLINGO,be0eb54e-bd00-4e93-b420-4ed34ca9682c,Alleya zevania Kiara,5,bagus buat lagi yang belajar bahasa asing,0,2026-04-26 11:40:01,5.141.11


In [ ]:
df = raw_df.copy()

# Pembersihan dasar
before = len(df)
df = df.drop_duplicates(subset=["review_id"]).reset_index(drop=True)
df = df.dropna(subset=["content", "score"])
df["content"] = df["content"].astype(str).str.strip()
df = df[df["content"].str.len() > 3].reset_index(drop=True)

df["at"] = pd.to_datetime(df["at"], errors="coerce")

print(f"Sebelum cleaning : {before:,}".replace(",", "."))
print(f"Setelah cleaning : {len(df):,}".replace(",", "."))
print("Distribusi skor:")
display(df["score"].value_counts().sort_index())

# Simpan ke CSV
ordered_cols = [
    "review_id", "app_id", "app_name", "user_name", "score", "content", "thumbs_up", "at", "app_version"
]
existing_cols = [c for c in ordered_cols if c in df.columns]
df[existing_cols].to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

print(f"\nFile CSV tersimpan di: {OUTPUT_CSV.resolve()}")
print(f"Jumlah sampel akhir: {len(df):,}".replace(",", "."))

Sebelum cleaning : 20.000
Setelah cleaning : 19.819
Distribusi skor:


,count
score,
1,570
2,206
3,508
4,1981
5,16554



File CSV tersimpan di: /content/data/reviews_duolingo.csv
Jumlah sampel akhir: 19.819


In [ ]:
df.sample(5, random_state=42)[["app_name", "score", "content"]]

,app_name,score,content
8188,DUOLINGO,5,seru banget jadi bisa belajar dari banyak bahasa seperti aku bisa bahasa inggris
3323,DUOLINGO,5,mem bantu melancarkan semua bahasa
7042,DUOLINGO,5,Menyenangkan dan memberikan saya pengalaman b. inggris saya jadi mudah menulis diary tanpa bisa di artikan oleh tema...
6858,DUOLINGO,5,bagus banget 😎
5027,DUOLINGO,5,sejauh ini saya suka entah untuk beberapa hari bagaimana tapi ini cocok untuk pemula soalnya seru
